In [ ]:
!pip install pdfplumber transformers torch

In [ ]:
import os
import glob
import re
import json
import pdfplumber
from transformers import pipeline
import zipfile
import pandas as pd

**Second step**

In [ ]:
pdf_file = "ayudas_25-26.pdf"

file_name = os.path.basename(pdf_file)
print(f"   -> Reading {file_name}...")

full_text = ""
with pdfplumber.open(pdf_file) as pdf:
    for page in pdf.pages:
        page_text = page.extract_text()
        if page_text:
            full_text += page_text + "\n"

print(f"Extracted {len(full_text)} characters.")

   -> Reading ayudas_25-26.pdf...
Extracted 167956 characters.


using a fined tuned bert model made for spanish (Question-Answering (QA) model) to ask the question and extract the data


# Information Extraction

## Question-Answering

In [ ]:
import torch
device = 0 if torch.cuda.is_available() else -1

# We use a Spanish model specifically trained to extract answers from text
qa_model = pipeline(
    "question-answering",
    model="mrm8488/bert-base-spanish-wwm-cased-finetuned-spa-squad2-es",
    device=device
)

extraction_fields = {
    "Academic_Year": "¿Para qué curso académico se convocan las becas?",
    "Issuing_Body": "¿Qué órgano/ministerio emite la convocatoria (organismo convocante)?",
    "BOE_Publication_Date": "¿En qué fecha se publica en el BOE esta convocatoria?",
    "Deadline_University": "¿Cuál es el plazo de solicitud para estudiantes universitarios (fecha límite)?",
    "Deadline_NonUniversity": "¿Cuál es el plazo de solicitud para estudiantes no universitarios (fecha límite)?",
    "Late_Application_Allowed": "¿Se permite presentar la solicitud fuera de plazo? ¿Hasta qué fecha y bajo qué circunstancias excepcionales?",
    "Application_Channel": "¿Cómo se presenta la solicitud (sede electrónica/telemática) y qué indica el BOE sobre el procedimiento?",


    "Total_Budget": "¿Cuál es la cantidad máxima o financiación de la convocatoria?",
    "Fixed_Income_Amount": "¿Cuál es la cuantía fija ligada a la renta?",
    "Residence_Amount": "¿Cuál es la cuantía fija ligada a la residencia?",
    "Minimum_Variable_Amount": "¿Cuál es la cuantía variable mínima?",

    # B) Estudios incluidos (ámbito de aplicación)
    "Eligible_Studies_NonUniversity": "¿Qué enseñanzas no universitarias están incluidas (lista de 'Enseñanzas comprendidas')?",
    "Eligible_Studies_University": "¿Qué estudios universitarios están incluidos (grado, máster, etc.)?",
    "Explicit_Exclusions": "¿Qué estudios se excluyen explícitamente (p. ej., doctorado, títulos propios u otros)?"
}

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForQuestionAnswering LOAD REPORT from: mrm8488/bert-base-spanish-wwm-cased-finetuned-spa-squad2-es
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
extracted_results = []

search_context = full_text


pdf_data = {}

for field_name, question in extraction_fields.items():
      result = qa_model(question=question, context=search_context)

      if result['score'] > 0.01:
          pdf_data[field_name] = result['answer']
      else:
          pdf_data[field_name] = "Not found confidently"


extracted_results.append(pdf_data)

df = pd.DataFrame(extracted_results)

df.to_csv("ayudas_25-26.csv", index=False)

display(df)

,Academic_Year,Issuing_Body,BOE_Publication_Date,Deadline_University,Deadline_NonUniversity,Late_Application_Allowed,Application_Channel,Total_Budget,Fixed_Income_Amount,Residence_Amount,Minimum_Variable_Amount,Eligible_Studies_NonUniversity,Eligible_Studies_University,Explicit_Exclusions
0,2025-2026,"Ministerio de Educación, Formación Profesional...",15 de junio,30 de septiembre de 2025,24 de marzo de 2025,31 de diciembre de 2025,vía\nSMS,concesión directa y\npago de una beca,"1.700,00 euros","2.700,00 euros",beca de matrícula,Artículo 63,arquitectura e\ningeniería,"créditos convalidados, reconocidos o adaptados"


## REGEX MERTHOD

In [ ]:
import re
import pandas as pd

MONTHS = r"(?:enero|febrero|marzo|abril|mayo|junio|julio|agosto|septiembre|setiembre|octubre|noviembre|diciembre)"
DATE_LONG = rf"\d{{1,2}}\s+de\s+{MONTHS}\s+de\s+\d{{4}}"

def _norm_spaces(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()

def _search(pattern, text, flags=re.IGNORECASE | re.DOTALL, group=1, default="Not found"):
    m = re.search(pattern, text, flags)
    return m.group(group).strip() if m else default

def extract_convocatoria_fields(full_text: str) -> dict:
    flat = _norm_spaces(full_text)

    out = {"Method": "RegEx"}

    # Academic year
    out["Academic_Year"] = _search(r"CURSO\s+ACAD[ÉE]MICO\s+(\d{4}-\d{4})", full_text)

    # Issuing body
    issuing = _search(
        r"RESOLUCI[ÓO]N\s+DE\s+LA\s+([^,]+?)\s*,\s*POR\s+LA\s+QUE\s+SE\s+CONVOCAN",
        full_text
    )
    if issuing != "Not found":
        out["Issuing_Body"] = issuing
    else:
        ministry = _search(r"\bMINISTERIO\s+DE\s+([A-ZÁÉÍÓÚÑ,\s]+)", full_text)
        out["Issuing_Body"] = ("Ministerio de " + ministry.title()) if ministry != "Not found" else "Not found"

    # BOE publication date
    res_date = _search(r"Resoluci[óo]n\s+de\s+(" + DATE_LONG + r")", full_text[:3000])
    if res_date != "Not found":
        out["BOE_Publication_Date"] = res_date
    else:
        out["BOE_Publication_Date"] = _search(r"FECHA\s*:\s*(\d{2}/\d{2}/\d{4})", full_text)

    # Deadlines
    uni_deadline = _search(r"A\)\s*El\s+(" + DATE_LONG + r")\s*,\s*inclusive\s*,\s*para\s+los\s+estudiantes\s+universitarios", full_text)
    nonuni_deadline = _search(r"B\)\s*El\s+(" + DATE_LONG + r")\s*,\s*inclusive\s*,\s*para\s+los\s+estudiantes\s+no\s+universitarios", full_text)

    win_start = _search(r"El\s+plazo\s+para\s+presentar\s+la\s+solicitud.*?desde\s+el\s+d[ií]a\s+(" + DATE_LONG + r")", full_text)
    win_end = _search(r"El\s+plazo\s+para\s+presentar\s+la\s+solicitud.*?hasta\s+el\s+(" + DATE_LONG + r")", full_text)

    if uni_deadline != "Not found" or nonuni_deadline != "Not found":
        out["Deadline_University"] = uni_deadline
        out["Deadline_NonUniversity"] = nonuni_deadline
    elif win_start != "Not found" and win_end != "Not found":
        window_text = f"Desde {win_start} hasta {win_end}"
        out["Deadline_University"] = window_text
        out["Deadline_NonUniversity"] = window_text
    else:
        out["Deadline_University"] = "Not found"
        out["Deadline_NonUniversity"] = "Not found"

    # Late applications
    late_until = _search(r"hasta\s+el\s+(" + DATE_LONG + r")\s+en\s+caso\s+de\s+fallecimiento", flat)
    if late_until != "Not found":
        out["Late_Application_Allowed"] = (
            f"Sí, hasta el {late_until}, en caso de fallecimiento del sustentador principal "
            "o por jubilación forzosa sobrevenida (no por edad reglamentaria)."
        )
    else:
        out["Late_Application_Allowed"] = "Not found"

    # Application channel
    url1 = _search(r"sede\s+electr[oó]nica\s+del\s+Departamento\s+en\s+la\s+direcci[oó]n\s+(https?://\S+|www\.\S+)", flat)
    url2 = _search(r"sede\s+electr[oó]nica\s+del\s+Departamento.*?(?:https?://\S+|www\.\S+)\s+o\s+en\s+(https?://\S+|www\.\S+)", flat)

    if url1 != "Not found":
        if url2 != "Not found":
            out["Application_Channel"] = f"Solicitud telemática en sede electrónica: {url1} y {url2}"
        else:
            out["Application_Channel"] = f"Solicitud telemática en sede electrónica: {url1}"
    else:
        out["Application_Channel"] = "Not found"

    # Total budget
    out["Total_Budget"] = _search(
        r"recursos\s+financieros\s+para\s+dicho\s+curso\s+ascender[aá]n\s+a\s+(\d{1,3}(?:\.\d{3})*(?:,\d+)?\s+millones\s+de\s+euros)",
        full_text
    )
    if out["Total_Budget"] == "Not found":
        cap = _search(r"no\s+podr[aá]\s+exceder\s+de\s+([\d\.]+,\d{2})\s*euros", full_text)
        out["Total_Budget"] = (cap + " euros (límite Art. 2)") if cap != "Not found" else "Not found"

    # Amounts
    rent_amt = _search(r"Cuant[ií]a\s+fija\s+ligada\s+a\s+la\s+renta\s+del\s+solicitante:\s*([\d\.]+,\d{2})\s*euros", full_text)
    out["Fixed_Income_Amount"] = (rent_amt + " €") if rent_amt != "Not found" else "Not found"

    res_amt = _search(r"Cuant[ií]a\s+fija\s+ligada\s+a\s+la\s+residencia\s+del\s+solicitante.*?:\s*([\d\.]+,\d{2})\s*euros", full_text)
    out["Residence_Amount"] = (res_amt + " €") if res_amt != "Not found" else "Not found"

    min_var = _search(r"importe\s+m[ií]nimo\s+ser[aá]\s+de\s+([\d\.]+,\d{2})\s*euros", full_text)
    out["Minimum_Variable_Amount"] = (min_var + " €") if min_var != "Not found" else "Not found"

    # Eligible studies + exclusions
    out["Eligible_Studies_NonUniversity"] = _search(
        r"Art[ií]culo\s+3\.\s*Ense[nñ]anzas\s+comprendidas\..*?"
        r"1\.\s*Ense[nñ]anzas\s+postobligatorias.*?:\s*(.*?)\s*"
        r"2\.\s*Ense[nñ]anzas",
        full_text
    )

    out["Eligible_Studies_University"] = _search(
        r"2\.\s*Ense[nñ]anzas.*?:\s*(.*?)(?:CAP[IÍ]TULO\s+II|Art[ií]culo\s+4)",
        full_text
    )

    out["Explicit_Exclusions"] = _search(
        r"No\s+se\s+incluyen\s+en\s+esta\s+convocatoria\s+(.*?)(?:\.|\n)",
        full_text
    )

    return out

result = extract_convocatoria_fields(full_text)
df_regex = pd.DataFrame([result])

df_regex.to_csv("ayudas_25-26-2.csv", index=False)

display(df_regex)
#print("=" * 80)
#print("📏 REGEX EXTRACTION RESULTS (Hard Numbers, Dates & Rules)")
#print("=" * 80)
#display(df_regex.T.rename(columns={0: "RegEx_Extracted_Value"}))

,Method,Academic_Year,Issuing_Body,BOE_Publication_Date,Deadline_University,Deadline_NonUniversity,Late_Application_Allowed,Application_Channel,Total_Budget,Fixed_Income_Amount,Residence_Amount,Minimum_Variable_Amount,Eligible_Studies_NonUniversity,Eligible_Studies_University,Explicit_Exclusions
0,RegEx,2025-2026,SECRETARÍA DE ESTADO DE EDUCACIÓN,Not found,Desde 24 de marzo de 2025 hasta 14 de mayo de ...,Desde 24 de marzo de 2025 hasta 14 de mayo de ...,"Sí, hasta el 31 de diciembre de 2025, en caso ...",Solicitud telemática en sede electrónica: http...,Not found,"1.700,00 €","1.700,00 €","60,00 €",a) Primer y segundo cursos de bachillerato.\nb...,a) Enseñanzas artísticas superiores.\n3\nTEC\n...,las becas para la realización de estudios corr...


## LLM Qwen 3B

In [ ]:
!pip -q install transformers accelerate bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.3 MB/s eta 0:00:00


In [ ]:
import re
import json
import gc
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "Qwen/Qwen2.5-3B-Instruct"  # or "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)
model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 2048)
    (layers): ModuleList(
      (0-35): 36 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=True)
          (k_proj): Linear(in_features=2048, out_features=256, bias=True)
          (v_proj): Linear(in_features=2048, out_features=256, bias=True)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (up_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((2048,), eps=1e-06)
    (ro

In [ ]:
def chunk_text(text: str, max_chars: int = 2600, overlap: int = 200):
    """
    Chunk by characters to ensure we process the full document
    Overlap helps keep context around cut boundaries.
    """
    text = re.sub(r"\s+", " ", text).strip()
    chunks = []
    i = 0
    while i < len(text):
        j = min(len(text), i + max_chars)
        chunks.append(text[i:j])
        if j == len(text):
            break
        i = max(0, j - overlap)
    return chunks

In [ ]:
TARGET_KEYS = {
    "Academic_Year": None,
    "Issuing_Body": None,
    "BOE_Publication_Date": None,
    "Deadline_University": None,
    "Deadline_NonUniversity": None,
    "Late_Application_Allowed": None,
    "Application_Channel": None,
    "Total_Budget": None,
    "Fixed_Income_Amount": None,
    "Residence_Amount": None,
    "Minimum_Variable_Amount": None,
    "Eligible_Studies_NonUniversity": None,
    "Eligible_Studies_University": None,
    "Explicit_Exclusions": None,
}

SYSTEM_LOCAL = (
    "Eres un extractor de información de convocatorias de becas del BOE.\n"
    "Devuelve SOLO JSON válido (sin texto extra, sin markdown).\n"
    "Usa EXACTAMENTE estas claves y estructura:\n"
    f"{json.dumps(TARGET_KEYS, ensure_ascii=False)}\n"
    "\n"
    "Reglas IMPORTANTES:\n"
    "- No inventes: si no aparece en el texto, usa null.\n"
    "- Para importes, respeta el formato que aparezca en el BOE (por ejemplo: '1.700,00 €' o '2.038,13 millones de euros').\n"
    "- Para plazos, si hay fecha límite única para todos, pon la misma fecha en Deadline_University y Deadline_NonUniversity.\n"
    "- Late_Application_Allowed debe explicar: si se permite, hasta qué fecha, y en qué circunstancias.\n"
    "- Como el usuario es UNIVERSITARIO, prioriza completar campos relevantes para universitarios, pero igualmente rellena todos si aparecen.\n"
    "- Evidence: incluye una cita corta (1-2 frases) tomada literalmente del texto para cada sección, si está disponible.\n"
)

In [ ]:
def parse_json_strict(text: str):
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        raise ValueError("No JSON object found in output.")
    snippet = text[start:end + 1]
    return json.loads(snippet)

In [ ]:
def extract_chunk_local(chunk: str, max_new_tokens: int = 450):
    prompt = SYSTEM_LOCAL + "\n\nTEXTO:\n" + chunk + "\n\nJSON:"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id,
        )

    gen_text = tokenizer.decode(
        output_ids[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )
    return parse_json_strict(gen_text)

In [ ]:
def merge_records(base: dict, new: dict) -> dict:
    def empty(x):
        return x is None or x == "" or x == [] or x == {}

    if not base:
        base = {}

    out = dict(base)

    for k, v in (new or {}).items():
        if isinstance(v, dict) and isinstance(out.get(k), dict):
            out[k] = merge_records(out[k], v)
            continue

        prev = out.get(k, None)
        if empty(prev) and not empty(v):
            out[k] = v
        else:
            out[k] = prev

    return out

In [ ]:
def extract_pdf_local(full_text: str):
    chunks = chunk_text(full_text, max_chars=2600, overlap=200)
    agg = {}

    for ch in chunks:
        try:
            data = extract_chunk_local(ch, max_new_tokens=450)
            agg = merge_records(agg, data)
        except Exception:
            pass

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Ensure all keys exist
    for k in TARGET_KEYS:
        if k not in agg:
            agg[k] = TARGET_KEYS[k]

    return agg

In [ ]:
def flatten_extraction(record: dict) -> dict:
    def safe(x):
        return "" if x is None else x

    evidence = record.get("Evidence", {}) or {}

    return {
        "Academic_Year": safe(record.get("Academic_Year")),
        "Issuing_Body": safe(record.get("Issuing_Body")),
        "BOE_Publication_Date": safe(record.get("BOE_Publication_Date")),
        "Deadline_University": safe(record.get("Deadline_University")),
        "Deadline_NonUniversity": safe(record.get("Deadline_NonUniversity")),
        "Late_Application_Allowed": safe(record.get("Late_Application_Allowed")),
        "Application_Channel": safe(record.get("Application_Channel")),
        "Total_Budget": safe(record.get("Total_Budget")),
        "Fixed_Income_Amount": safe(record.get("Fixed_Income_Amount")),
        "Residence_Amount": safe(record.get("Residence_Amount")),
        "Minimum_Variable_Amount": safe(record.get("Minimum_Variable_Amount")),
        "Eligible_Studies_NonUniversity": safe(record.get("Eligible_Studies_NonUniversity")),
        "Eligible_Studies_University": safe(record.get("Eligible_Studies_University")),
        "Explicit_Exclusions": safe(record.get("Explicit_Exclusions"))
    }

In [ ]:
local_json = extract_pdf_local(full_text)  # full_text = entire PDF as text
row = flatten_extraction(local_json)
df_llm = pd.DataFrame([row])
df_llm.to_csv("ayudas_25-26-llm.csv", index=False)
df_llm

,Academic_Year,Issuing_Body,BOE_Publication_Date,Deadline_University,Deadline_NonUniversity,Late_Application_Allowed,Application_Channel,Total_Budget,Fixed_Income_Amount,Residence_Amount,Minimum_Variable_Amount,Eligible_Studies_NonUniversity,Eligible_Studies_University,Explicit_Exclusions
0,2025-2026,"MINISTERIO DE EDUCACIÓN, FORMACIÓN PROFESIONAL...",15 de junio,2025-04-30,2025-04-30,No se permite la solicitud tardía.,Sólo se puede solicitar a través de la web del...,"1.700,00 €","1.700,00 €","1.700,00 €",no especificado,Estudios postobligatorios de nivel no universi...,Estudios postobligatorios de nivel universitario,no especificado
